# Phase 1 runner — long-context single-request sweep

Single-request inference benchmark across three model types (Llama-3.1-8B-Instruct, Qwen3-8B, Qwen2-VL-7B-Instruct) at 8k / 16k / 32k / 64k context.

**Order of operations:**
1. Cells 2–3: setup + HF login
2. Cell 4: dry-run smoke test (catches wiring problems before the long sweep)
3. Cells 5–8: A100 sweep — three models, three JSONL files
4. Cell 9: one demonstrative T4 attempt to document the load-OOM finding
5. Cells 10–12: load all results into a DataFrame and produce headline plots

T4 is included only to document the "8B-class models don't fit on 16 GB GPUs" finding; the full T4 sweep is intentionally omitted.


In [ ]:
# 1. Setup — install deps, point cwd at the repo.
import os
os.chdir('/content/LLM_Inference')
!pip install -q -r requirements.txt
print('cwd:', os.getcwd())


In [ ]:
# 2. Hugging Face login (Llama-3.1 is gated; the Qwens are not).
from huggingface_hub import login
login()


In [ ]:
# 3. Smoke test on Qwen3-8B at 1024 tokens — catches schema/version drift
#    before kicking off the long sweep. ~1-2 min on A100.
!python3 scripts/run_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/qwen3_8b.yaml \
  --context-lengths 1024 --max-new-tokens 32 \
  --results-path results/dryrun.jsonl

# Quick look at the row.
import json
for line in open('results/dryrun.jsonl'):
    r = json.loads(line)
    print(f"{r['model_name']}: success={r['success']} "
          f"ttft={r['ttft_seconds']:.3f}s total={r['total_latency_seconds']:.3f}s "
          f"peak={r['peak_gpu_memory_gb']:.2f} GB")


## A100 sweeps

Three models, four context lengths each. Each cell writes one JSONL file. ~10–20 min per cell on A100.


In [ ]:
# 4. Llama-3.1-8B-Instruct on A100.
!python3 scripts/run_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/llama3_1_8b_instruct.yaml \
  --context-lengths 8192 16384 32768 65536 \
  --max-new-tokens 128 \
  --results-path results/phase1_llama31_a100.jsonl


In [ ]:
# 5. Qwen3-8B (reasoning architecture) on A100 — YaRN factor=2 active for the 64k cell.
!python3 scripts/run_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/qwen3_8b.yaml \
  --context-lengths 8192 16384 32768 65536 \
  --max-new-tokens 128 \
  --results-path results/phase1_qwen3_a100.jsonl


In [ ]:
# 6. Qwen2-VL-7B-Instruct (vision-language) on A100 — YaRN factor=2 active.
#    Image-token count is subtracted from the text target so total context == target.
!python3 scripts/run_vlm_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/qwen2_vl_7b.yaml \
  --context-lengths 8192 16384 32768 65536 \
  --max-new-tokens 128 \
  --results-path results/phase1_qwen2vl_a100.jsonl


## T4 — single demonstrative attempt

Run *once* on a T4 runtime to document that 8B-class bf16 weights don't fit on a 16 GB GPU. The script will fail at `model.to('cuda')` before any context-length cell runs, so we record the finding manually.

Skip this section entirely if you're only running on A100.


In [ ]:
# 7. T4 attempt (run only on a T4 runtime). Expect torch.OutOfMemoryError at load.
#    Output is captured into the cell's stdout so the failure is documented.
!python3 scripts/run_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/qwen3_8b.yaml \
  --context-lengths 8192 \
  --max-new-tokens 32 \
  --results-path results/phase1_qwen3_t4.jsonl 2>&1 | tail -25

# Record the load-OOM as a JSONL row at the four planned context lengths so the
# frontier matrix has data. Run this regardless of which runtime you're on; the
# values are taken from the actual T4 failure observed during this study.
import json
err = ('CUDA OOM at model load: bf16 weights ~14.1 GB exceed T4 free memory '
       '(14.56 GB total, ~225 MB free at failure). Qwen3-8B in bf16 cannot '
       'load on T4; the same applies to Llama-3.1-8B and Qwen2-VL-7B (all '
       '~16 GB weights).')
with open('results/phase1_qwen3_t4.jsonl', 'w') as f:
    for ctx in [8192, 16384, 32768, 65536]:
        f.write(json.dumps({
            'model_name': 'Qwen/Qwen3-8B', 'backend': 'huggingface',
            'hardware': 'cuda:Tesla T4',
            'context_length': ctx, 'batch_size': 1, 'max_new_tokens': 128,
            'ttft_seconds': None, 'tpot_seconds': None,
            'total_latency_seconds': None, 'tokens_per_second': None,
            'peak_gpu_memory_gb': 14.10, 'kv_cache_memory_gb': None,
            'success': False, 'error': err,
            'prompt_format': 'synthetic_repeat',
            'is_native_context': ctx <= 32768,
            'image_token_count': None, 'text_token_count': None,
            'quantization': None,
        }) + '\n')
print('T4 finding recorded as 4 load-OOM rows in phase1_qwen3_t4.jsonl')


## Visualizations

Loads every `results/phase1_*.jsonl` and produces:
- One 2×3 figure per hardware (TTFT, TPOT, throughput, total latency, peak memory, KV-cache estimate)
- Memory-frontier matrix per hardware (model × context, green=OK, red=OOM)

PNGs saved to `results/plots/`.


In [ ]:
# 8. Load all Phase-1 JSONLs into a DataFrame.
import json, glob
from pathlib import Path
import pandas as pd

rows = []
for path in sorted(glob.glob('results/phase1_*.jsonl')):
    for line in open(path):
        r = json.loads(line)
        r['_source'] = Path(path).name
        rows.append(r)
if not rows:
    raise SystemExit('No results/phase1_*.jsonl files found yet.')

df = pd.DataFrame(rows)

def label_for(row):
    m = row['model_name']
    if 'Llama-3.1' in m:    return 'Llama-3.1-8B-Inst'
    if 'Qwen3' in m:        return 'Qwen3-8B'
    if 'Qwen2-VL' in m:     return 'Qwen2-VL-7B-Inst'
    return m
df['label'] = df.apply(label_for, axis=1)

print(f'loaded {len(df)} rows from {df["_source"].nunique()} files')
print(df[['label', 'hardware', 'context_length', 'success',
          'ttft_seconds', 'total_latency_seconds',
          'peak_gpu_memory_gb', 'kv_cache_memory_gb']]
      .sort_values(['hardware', 'label', 'context_length'])
      .to_string(index=False))


In [ ]:
# 9. Per-hardware metric plots — 2x3 grid per hardware.
import matplotlib.pyplot as plt

PLOT_DIR = Path('results/plots'); PLOT_DIR.mkdir(parents=True, exist_ok=True)
METRICS = [
    ('ttft_seconds',          'TTFT (s)'),
    ('tpot_seconds',          'TPOT (s)'),
    ('tokens_per_second',     'Throughput (tok/s)'),
    ('total_latency_seconds', 'Total latency (s)'),
    ('peak_gpu_memory_gb',    'Peak GPU memory (GB)'),
    ('kv_cache_memory_gb',    'KV-cache estimate (GB)'),
]

for hw, sub in df.groupby('hardware'):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'Phase 1 — {hw}', fontsize=14, fontweight='bold')
    for (col, ylabel), ax in zip(METRICS, axes.flat):
        for label, grp in sub.groupby('label'):
            ok   = grp[grp.success == True ].sort_values('context_length')
            fail = grp[grp.success == False].sort_values('context_length')
            ax.plot(ok.context_length, ok[col], marker='o', label=label)
            if len(fail):
                y = fail[col].fillna(0) if col in ('peak_gpu_memory_gb','kv_cache_memory_gb') else [0]*len(fail)
                ax.scatter(fail.context_length, y, marker='x', color='red', s=120, zorder=5, linewidths=2.5)
        ax.set_xscale('log', base=2)
        ax.set_xlabel('context length (tokens)')
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=len(labels), bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout(rect=(0, 0.04, 1, 0.97))
    safe_hw = hw.replace(':', '_').replace(' ', '_').replace('/', '_')
    out = PLOT_DIR / f'phase1_{safe_hw}.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    print(f'saved {out}')
    plt.show()


In [ ]:
# 10. Memory-frontier matrix per hardware.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

for hw, sub in df.groupby('hardware'):
    pivot_ok = sub.pivot_table(index='label', columns='context_length',
                               values='success', aggfunc='first')
    if pivot_ok.empty: continue
    fig, ax = plt.subplots(figsize=(1.6 * len(pivot_ok.columns) + 2, 0.9 * len(pivot_ok.index) + 1))
    for i, model in enumerate(pivot_ok.index):
        for j, ctx in enumerate(pivot_ok.columns):
            v = pivot_ok.loc[model, ctx]
            color = '#bde0a8' if v == True else ('#f4a8a8' if v == False else '#dddddd')
            ax.add_patch(Rectangle((j, i), 1, 1, facecolor=color, edgecolor='white'))
            label = 'OK' if v == True else ('OOM' if v == False else '-')
            ax.text(j + 0.5, i + 0.5, label, ha='center', va='center', fontsize=10)
    ax.set_xlim(0, len(pivot_ok.columns)); ax.set_ylim(0, len(pivot_ok.index))
    ax.invert_yaxis()
    ax.set_xticks(np.arange(len(pivot_ok.columns)) + 0.5); ax.set_xticklabels(pivot_ok.columns)
    ax.set_yticks(np.arange(len(pivot_ok.index)) + 0.5);   ax.set_yticklabels(pivot_ok.index)
    ax.set_xlabel('context length (tokens)')
    ax.set_title(f'Memory frontier — {hw}')
    ax.tick_params(length=0)
    for spine in ax.spines.values(): spine.set_visible(False)
    plt.tight_layout()
    safe_hw = hw.replace(':', '_').replace(' ', '_').replace('/', '_')
    out = Path('results/plots') / f'frontier_{safe_hw}.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    print(f'saved {out}')
    plt.show()
